In [ ]:
import pandas as pd 


In [ ]:
pools_colleted_from_gmaps = pd.read_parquet("/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany/germany_places.parquet")
pools_colleted_from_gmaps.info()

In [ ]:
#pools_colleted_from_gmaps[['latitude', 'longitude']].to_csv("/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany/germany_places_coordinates.csv", index=False)

In [ ]:
pools_colleted_within = pd.read_parquet("/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany/germany_places_within.parquet")
pools_colleted_within.info()

In [ ]:
pools_colleted_from_gmaps[~pools_colleted_from_gmaps['unique_identifier'].isin(pools_colleted_within['unique_identifier'])][['place_name', 'address', 'latitude', 'longitude', 'primary_place_type', 'place_subtype']].head(54)

## filtering to select the swimming pools

In [ ]:
"""
pool_filter.py
--------------
Filters a DataFrame to identify public swimming pool venues based on
primary_place_type and sub_place_types columns.

Usage in notebook or script:
    from pool_filter import filter_pools

    filtered_df = filter_pools(df)
    # or with custom column names:
    filtered_df = filter_pools(df, primary_col='my_primary', sub_col='my_subtypes')
"""

import ast
import pandas as pd


# ---------------------------------------------------------------------------
# Place type lists
# ---------------------------------------------------------------------------

# Primary tag alone is sufficient to classify as a pool venue
STRICT_PLACE_TYPES = {
    'beach',
    'indoor swimming pool',
    'lake',
    'lake shore swimming area',
    'lido',
    'outdoor bath',
    'outdoor swimming pool',
    'public bath',
    'public bathroom',
    'public beach',
    'public swimming pool',
    'super public bath',
    'swimming facility',
    'swimming lake',
    'swimming pool',
    'thermal baths',
}

# Primary tag qualifies only if at least one subtype is in STRICT_PLACE_TYPES
RELAXED_PLACE_TYPES = {
    'aquatic center',
    'leisure center',
    'recreation center',
    'sauna',
    'spa',
    'spa and health club',
    'sports complex',
    'swimming facility',
    'swimming lake',
    'swimming pool',
    'thermal baths',
    'water park',
}

# Primary tag qualifies even with no subtypes (assumed to have a pool)
ASSUMED_POOL_PRIMARIES = {
    'aquatic center',
    'sports complex',
    'recreation center',
    'water park',
}



# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _parse_subtypes(val):
    """Parse a subtype value into a list of lowercase strings."""
    if val is None:
        return []
    if isinstance(val, (list, tuple, set)):
        flattened = []
        for item in val:
            if isinstance(item, (list, tuple, set)):
                flattened.extend(_parse_subtypes(item))
                continue
            item_text = str(item).strip()
            if not item_text:
                continue
            if len(item_text) >= 2 and ((item_text[0] == item_text[-1] == '"') or (item_text[0] == item_text[-1] == "'")):
                item_text = item_text[1:-1].strip()
            if item_text.startswith('[') and item_text.endswith(']'):
                flattened.extend(_parse_subtypes(item_text))
                continue
            flattened.append(item_text.lower())
        return flattened
    if hasattr(val, 'tolist') and not isinstance(val, str):
        return _parse_subtypes(val.tolist())
    try:
        if pd.isna(val):
            return []
    except (TypeError, ValueError):
        pass

    text = str(val).strip()
    if not text:
        return []
    if len(text) >= 2 and ((text[0] == text[-1] == '"') or (text[0] == text[-1] == "'")):
        text = text[1:-1].strip()

    try:
        parsed = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return []

    if isinstance(parsed, str):
        return _parse_subtypes(parsed)
    if isinstance(parsed, (list, tuple, set)):
        return _parse_subtypes(list(parsed))
    return []


def _is_pool(primary_tag, sub_tags_parsed):
    """
    Returns True if the place is a public swimming pool venue.

    Rules (evaluated in order):
      1. Primary tag is in STRICT_PLACE_TYPES → True
      2. Primary tag is in RELAXED_PLACE_TYPES and at least one subtype
         is in STRICT_PLACE_TYPES → True
      4. Primary tag is in ASSUMED_POOL_PRIMARIES and no subtypes → True
      5. Otherwise → False
    """
    if pd.isna(primary_tag):
        return False

    primary = primary_tag.strip().lower()

    # Rule 1: strict primary match
    if primary in STRICT_PLACE_TYPES:
        return True

    # Rule 2: relaxed primary + at least one strict subtype
    if primary in RELAXED_PLACE_TYPES:
        if any(s in STRICT_PLACE_TYPES for s in sub_tags_parsed):
            return True

    # Rule 3: assumed pool primary with no subtypes
    if primary in ASSUMED_POOL_PRIMARIES and not sub_tags_parsed:
        return True

    return False


# ---------------------------------------------------------------------------
# Main function
# ---------------------------------------------------------------------------

def filter_pools(
    df,
    primary_col='primary_place_type',
    sub_col='place_subtype',
    keep_mask_col=False,
):
    """
    Filter a DataFrame to rows that are likely public swimming pool venues.

    Parameters
    ----------
    df : pd.DataFrame
    primary_col : str
        Column name for the primary place type.
    sub_col : str
        Column name for the sub place types (stringified list or actual list).
    keep_mask_col : bool
        If True, adds a boolean column 'is_pool' to the returned DataFrame.

    Returns
    -------
    pd.DataFrame
        Filtered DataFrame (and optionally with 'is_pool' column).
    """
    if primary_col not in df.columns:
        raise ValueError(f"Column '{primary_col}' not found in DataFrame.")
    if sub_col not in df.columns:
        raise ValueError(f"Column '{sub_col}' not found in DataFrame.")

    parsed_subs = df[sub_col].apply(_parse_subtypes)
    mask = [
        _is_pool(primary, subs)
        for primary, subs in zip(df[primary_col], parsed_subs)
    ]

    result = df[mask].copy()

    if keep_mask_col:
        result['is_pool'] = True

    return result

In [ ]:
pools_colleted_within['primary_place_type'] = pools_colleted_within['primary_place_type'].str.lower()

In [ ]:
filtered_pools = filter_pools(pools_colleted_within, primary_col='primary_place_type', sub_col='place_subtype', keep_mask_col=True)
filtered_pools.info()

## Which are still lacking reviews

In [ ]:
all_reviews_collected = pd.read_parquet("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/processed_data/final_all_reviews_updated_30042026.parquet")
all_reviews_collected.info()

In [ ]:
#Select reviews for the filtered pools using unique_identifier or place_id
filtered_reviews = all_reviews_collected[(all_reviews_collected['unique_identifier'].isin(filtered_pools['unique_identifier']))|(all_reviews_collected['place_id'].isin(filtered_pools['place_id']))]
filtered_reviews.info()

In [ ]:
#only tex reviews
text_reviews = filtered_reviews[filtered_reviews['org_review'].notna() & (filtered_reviews['org_review'] != '')].reset_index(drop=True)
text_reviews.info()

In [ ]:
print('filtered_pools columns:', filtered_pools.columns.tolist())
print('all_reviews_collected columns:', all_reviews_collected.columns.tolist())


In [ ]:
# Fill missing review unique_identifier from filtered_pools using place_id
pid_to_uid = (
    filtered_pools
    .dropna(subset=['place_id', 'unique_identifier'])
    .drop_duplicates(subset=['place_id'])
    .set_index('place_id')['unique_identifier']
)

reviews_for_count = all_reviews_collected.copy()
missing_uid = reviews_for_count['unique_identifier'].isna() | reviews_for_count['unique_identifier'].astype(str).str.strip().eq('')
reviews_for_count.loc[missing_uid, 'unique_identifier'] = reviews_for_count.loc[missing_uid, 'place_id'].map(pid_to_uid)

# Keep only reviews that belong to pools in filtered_pools (by unique_identifier OR place_id)
pool_uids = set(filtered_pools['unique_identifier'].dropna().astype(str))
pool_pids = set(filtered_pools['place_id'].dropna().astype(str))

review_uids = reviews_for_count['unique_identifier'].astype(str)
review_pids = reviews_for_count['place_id'].astype(str)

filtered_reviews = reviews_for_count.loc[
    review_uids.isin(pool_uids) | review_pids.isin(pool_pids)
].copy()

# One pool row per place_id for reporting
pool_base = (
    filtered_pools
    .drop_duplicates(subset=['place_id'])
    .loc[:, ['place_id', 'place_name', 'primary_place_type', 'total_reviews']]
    .copy()
)
pool_base['total_reviews'] = pd.to_numeric(pool_base['total_reviews'], errors='coerce').fillna(0).astype(int)

# Count unique review_id per pool in collected reviews
reviews_collected = (
    filtered_reviews
    .dropna(subset=['review_id'])
    .groupby('place_id')['review_id']
    .nunique()
    .rename('reviews_collected')
)

review_coverage_table = (
    pool_base
    .merge(reviews_collected, on='place_id', how='left')
    .fillna({'reviews_collected': 0})
)
review_coverage_table['reviews_collected'] = review_coverage_table['reviews_collected'].astype(int)
review_coverage_table['remaining_reviews'] = (
    review_coverage_table['total_reviews'] - review_coverage_table['reviews_collected']
).clip(lower=0)

review_coverage_table = review_coverage_table.loc[:, [
    'place_name', 'primary_place_type', 'total_reviews', 'reviews_collected', 'remaining_reviews'
]].sort_values(['remaining_reviews', 'total_reviews'], ascending=False)

review_coverage_table


In [ ]:
from IPython.display import display

# Summary of missing reviews across pools
places_missing_a_review = int((review_coverage_table['remaining_reviews'] > 0).sum())
total_remaining_reviews = int(review_coverage_table['remaining_reviews'].sum())

review_gap_summary = pd.DataFrame({
    'metric': [
        'places_missing_at_least_one_review',
        'total_remaining_reviews',
    ],
    'value': [
        places_missing_a_review,
        total_remaining_reviews,
    ],
})

# Distribution of remaining reviews across pools that still have a gap
remaining_reviews_distribution = (
    review_coverage_table.loc[review_coverage_table['remaining_reviews'] > 0, 'remaining_reviews']
    .value_counts()
    .sort_index(ascending=False)
    .rename_axis('remaining_reviews')
    .reset_index(name='number_of_places')
)

display(review_gap_summary)
display(remaining_reviews_distribution)


In [ ]:
# Calculate percentage of missing reviews per place
review_coverage_with_pct = review_coverage_table.copy()
review_coverage_with_pct['missing_review_pct'] = (
    (review_coverage_with_pct['remaining_reviews'] / review_coverage_with_pct['total_reviews'] * 100)
    .round(1)
)

# Create percentage bins: 0-10%, 10-20%, 20-30%, etc.
pct_bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
pct_labels = ['0-10%', '10-20%', '20-30%', '30-40%', '40-50%', '50-60%', '60-70%', '70-80%', '80-90%', '90-100%']

review_coverage_with_pct['pct_range'] = pd.cut(
    review_coverage_with_pct['missing_review_pct'],
    bins=pct_bins,
    labels=pct_labels,
    right=False
)

# Distribution of places by percentage of missing reviews
places_by_missing_pct = (
    review_coverage_with_pct.loc[review_coverage_with_pct['remaining_reviews'] > 0, 'pct_range']
    .value_counts()
    .sort_index()
    .rename_axis('missing_review_percentage')
    .reset_index(name='number_of_places')
)

display(places_by_missing_pct)

In [ ]:
# Pareto analysis: cumulative contribution to missing reviews
places_with_gaps = review_coverage_table[review_coverage_table['remaining_reviews'] > 0].copy()
places_with_gaps = places_with_gaps.sort_values('remaining_reviews', ascending=False).reset_index(drop=True)

total_missing = places_with_gaps['remaining_reviews'].sum()
places_with_gaps['cumulative_missing'] = places_with_gaps['remaining_reviews'].cumsum()
places_with_gaps['cumulative_pct'] = (places_with_gaps['cumulative_missing'] / total_missing * 100).round(1)
places_with_gaps['place_rank'] = range(1, len(places_with_gaps) + 1)

# Find cutoffs: how many places needed to reach 10%, 20%, 30%... 90% of missing reviews
thresholds = [10, 20, 30, 40, 50, 60, 70, 80, 90]
cutoff_data = []

for threshold in thresholds:
    first_reach = places_with_gaps[places_with_gaps['cumulative_pct'] >= threshold].iloc[0] if any(places_with_gaps['cumulative_pct'] >= threshold) else None
    if first_reach is not None:
        cutoff_data.append({
            'cumulative_pct_threshold': f'{threshold}%',
            'number_of_places': int(first_reach['place_rank']),
            'total_missing_reviews_covered': int(first_reach['cumulative_missing']),
        })

cumulative_analysis = pd.DataFrame(cutoff_data)
display(cumulative_analysis)

In [ ]:
import math

# Table for pools still missing at least one review
reviews_for_count['retrieved_time'] = pd.to_datetime(reviews_for_count['retrieved_time'], errors='coerce')

next_key_by_place = (
    reviews_for_count
    .dropna(subset=['place_id'])
    .sort_values(['place_id', 'retrieved_time'])
    .groupby('place_id', as_index=True)
    .tail(1)
    .set_index('place_id')['next_key']
)

pool_review_gap_table = (
    pool_base
    .merge(reviews_collected, on='place_id', how='left')
    .fillna({'reviews_collected': 0})
)
pool_review_gap_table['reviews_collected'] = pool_review_gap_table['reviews_collected'].astype(int)
pool_review_gap_table['remaining_reviews'] = (
    pool_review_gap_table['total_reviews'] - pool_review_gap_table['reviews_collected']
).clip(lower=0)
pool_review_gap_table = pool_review_gap_table.loc[pool_review_gap_table['remaining_reviews'] > 0].copy()

pool_review_gap_table['expected_jsons'] = (pool_review_gap_table['total_reviews'].astype(float) / 10).apply(math.ceil).astype(int)
pool_review_gap_table['collected_jsons'] = (pool_review_gap_table['reviews_collected'].astype(float) / 10).apply(math.ceil).astype(int)
pool_review_gap_table['unique_identifier'] = pool_review_gap_table['place_id'].map(
    filtered_pools.drop_duplicates(subset=['place_id']).set_index('place_id')['unique_identifier']
)
pool_review_gap_table['next_key'] = pool_review_gap_table['place_id'].map(next_key_by_place)

pool_review_gap_table = pool_review_gap_table.loc[:, [
    'place_id',
    'unique_identifier',
    'expected_jsons',
    'collected_jsons',
    'next_key',
]].sort_values(['expected_jsons', 'collected_jsons'], ascending=False)

pool_review_gap_table


In [ ]:
last_key_file = pd.read_csv('/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/review_counts_with_latest_next_key.csv')
last_key_file.info()

In [ ]:
pool_review_gap_table[pool_review_gap_table['expected_jsons'] > pool_review_gap_table['collected_jsons']][['next_key']]

In [ ]:
# Populate `next_key` in `pool_review_gap_table` from `last_key_file`
# - takes the latest non-null `next_key` per place_id from `last_key_file`
# - maps it onto `pool_review_gap_table`, replacing any previous values

# Ensure the required DataFrames exist
if 'last_key_file' not in globals():
    raise NameError('`last_key_file` not found in notebook namespace')
if 'pool_review_gap_table' not in globals():
    raise NameError('`pool_review_gap_table` not found in notebook namespace')

# Clean next_key in last_key_file and pick the last non-null next_key per place_id
last_key_file['next_key'] = last_key_file.get('next_key')
# Normalize strings and remove NULs
last_key_file['next_key'] = last_key_file['next_key'].apply(lambda x: x.replace('\x00', '') if isinstance(x, str) else x)

# Build mapping: take the most recent next_key per place_id where next_key is not null
next_key_by_place = (
    last_key_file.loc[last_key_file['next_key'].notna(), ['place_id', 'next_key']]
    .drop_duplicates(subset=['place_id', 'next_key'], keep='last')
    .groupby('place_id')['next_key']
    .last()
)

# Map into pool_review_gap_table
pool_review_gap_table['next_key'] = pool_review_gap_table['place_id'].map(next_key_by_place)

# Show a sample of changed rows
print('Mapped next_key for', pool_review_gap_table['next_key'].notna().sum(), 'places')
print(pool_review_gap_table[['place_id', 'next_key']].head(10))

In [ ]:
still_missing_reviews =pool_review_gap_table[pool_review_gap_table['expected_jsons'] > pool_review_gap_table['collected_jsons']].reset_index(drop=True)
still_missing_reviews

In [ ]:
still_missing_reviews.next_key.nunique()

In [ ]:
still_missing_reviews[still_missing_reviews['next_key'].notnull()].reset_index(drop=True).to_csv('/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany/pool_review_gaps_with_next_keys.csv', index=False)

In [ ]:
still_missing_reviews[still_missing_reviews['next_key'].isna()].reset_index(drop=True)#.to_csv('/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany/pool_review_gaps_without_next_keys.csv', index=False)